<a href="https://colab.research.google.com/github/kritirakheja/elizabeth-bennet-llm/blob/main/01_finetuning_elizabeth_bennet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Finetuning a LLM on a Fictional Charcter

- Link to HuggingFace dataset: hugging face dataset
- Link to HuggingFace adapters: finetuned model

##Set Up

In [ ]:
!pip install unsloth -q

In [ ]:
#HF Key

from google.colab import userdata
userdata.get('hf_token')

##Data: from novel to Q&A pairs

This dataset was created using unsloth recipes. Read more about it here [link blog]

In [ ]:
from datasets import load_dataset
from google.colab import userdata

# Load the main dataset
dataset = load_dataset("KritiR/Elizabeth_question_answer", "data", split="train", token=userdata.get('hf_token'))

In [ ]:
print(dataset)

In [ ]:
dataset['llm_structured_1'][0]

##Load Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

##Format Training Data

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

In [ ]:
import json

def to_conversations(example):
    qa = example["llm_structured_1"]
    if isinstance(qa, str):
        qa = json.loads(qa)
    return {
        "conversations": [
            {"role": "user",      "content": qa["question"]},
            {"role": "assistant", "content": qa["answer"]},
        ]
    }

dataset = dataset.map(to_conversations)

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    bos = tokenizer.bos_token
    texts = [t[len(bos):] if bos and t.startswith(bos) else t for t in texts]
    return { "text": texts }

In [ ]:
# dividing the data into training and validation set

split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]
print(len(train_ds), "train /", len(val_ds), "validation")

In [ ]:
train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func, batched=True)

In [ ]:
#example

train_ds["text"][0]

##Training

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset=val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        max_steps = -1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        eval_strategy="steps",
        eval_steps=5
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

for log in trainer.state.log_history:
    print(log)

##Inference

In [ ]:
# inference with fine tuned model

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "What do you think of your Mr Dracy?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.7,
    min_p = 0.05,
    eos_token_id = [tokenizer.eos_token_id, eot_id],
    )

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
#inference with base model

from unsloth.chat_templates import get_chat_template

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template = "llama-3.1")
FastLanguageModel.for_inference(base_model)

base_eot_id = base_tokenizer.convert_tokens_to_ids("<|eot_id|>")

base_inputs = base_tokenizer.apply_chat_template(
    messages,  # same prompt used for the fine-tuned model above, for a fair comparison
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

base_outputs = base_model.generate(
    input_ids = base_inputs,
    max_new_tokens = 128,
    use_cache = True,
    temperature = 0.7,
    min_p = 0.05,
    eos_token_id = [base_tokenizer.eos_token_id, base_eot_id],
)

print(base_tokenizer.decode(base_outputs[0], skip_special_tokens = True))

##Push to Hugging Face

In [ ]:
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

Read about how to evlaue a fine-tuned character model here [add link].